# Phase 3: Intelligent Dispatch Training (DQN)
**Objective:** Train the Deep Q-Network to optimize ambulance dispatching across Kigali.

This notebook initializes the SUMO environment, defines the Reinforcement Learning state-space, and executes the training loop. We utilize an $\epsilon$-greedy exploration strategy so the agent learns to balance random discovery with exploiting its trained neural network. 

Crucially, this loop is **strictly resumable**. It will automatically detect existing checkpoint weights and resume training from the last saved epoch to prevent data loss.

In [1]:
import sys
import json
import math
import logging
import numpy as np
from pathlib import Path

# Add project root to path
sys.path.append(str(Path.cwd().parent))

from src.environment.manager import SimulationManager
from src.environment.hospital import Hospital
from src.agents.dispatch_dqn import DispatchAgent

# Configure basic logging for the notebook
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Define paths
net_path = Path("../data/processed/kigali.net.xml")
route_path = Path("../data/processed/kigali_traffic.rou.xml")
incidents_path = Path("../data/processed/incidents_seed42.json")
model_save_path = Path("../models/dqn_dispatch_v1.pt")

## 1. The RL Environment Wrapper
We create a lightweight wrapper class to handle the state extraction and reward calculation at every step. To keep the training loop fast during these initial epochs, we use a Euclidean distance proxy for drive times (assuming an average ambulance speed of 15 m/s or ~54 km/h). The dynamic hospital queuing wait times are calculated exactly using our Phase 2 logic.

In [2]:
class EMSEnvironment:
    def __init__(self, incidents_path: Path):
        # Load the deterministic incident schedule
        with open(incidents_path, 'r') as f:
            self.incident_schedule = json.load(f)
            
        # Re-initialize the 8 Hospitals and 12 Ambulances from Phase 2
        self.hospitals = [
            Hospital("CHUK", "CHUK", "edge", 20, 1.5), Hospital("KFH", "KFH", "edge", 10, 2.0),
            Hospital("RMH", "RMH", "edge", 15, 1.5), Hospital("KIB", "KIB", "edge", 8, 1.2),
            Hospital("NYA", "NYA", "edge", 8, 1.2), Hospital("KAC", "KAC", "edge", 8, 1.2),
            Hospital("MAS", "MAS", "edge", 8, 1.2), Hospital("MUH", "MUH", "edge", 6, 1.2)
        ]
        
        # Base locations for the fleet
        self.bases = [
            (8777.2, 13225.8), (8777.2, 13225.8), (8777.2, 13225.8), # CHUK
            (16910.0, 10915.7), (16910.0, 10915.7),                  # RMH
            (12618.8, 13298.9), (12618.8, 13298.9),                  # KFH
            (15566.8, 15008.3), (6892.3, 8574.0),                    # KIB, NYA
            (11003.5, 13672.4), (24042.0, 7497.3), (8554.1, 13446.8) # KAC, MAS, MUH
        ]
        self.reset()

    def reset(self):
        """Resets the environment for a new episode."""
        for h in self.hospitals:
            h.reset()
        
        # Fleet state: [x, y, available_flag] for 12 ambulances
        self.fleet = [{"x": bx, "y": by, "available": 1.0} for bx, by in self.bases]
        self.current_incident_idx = 0
        return self._get_state()

    def _get_state(self):
        """Constructs the 47-dimensional state vector."""
        if self.current_incident_idx >= len(self.incident_schedule):
            return np.zeros(47, dtype=np.float32) # Terminal state
            
        incident = self.incident_schedule[self.current_incident_idx]
        
        state = []
        # 1. Fleet State (36 values)
        for amb in self.fleet:
            state.extend([amb["x"], amb["y"], amb["available"]])
        # 2. Hospital State (8 values)
        for h in self.hospitals:
            state.append(h.current_queue)
        # 3. Incident State (3 values)
        state.extend([incident["x"], incident["y"], incident["severity"]])
        
        return np.array(state, dtype=np.float32)

    def step(self, action_idx: int):
        """Executes the dispatch action and calculates the reward."""
        incident = self.incident_schedule[self.current_incident_idx]
        selected_amb = self.fleet[action_idx]
        
        # 1. Calculate Drive Time (Distance / 15 meters per second)
        dist = math.sqrt((selected_amb["x"] - incident["x"])**2 + (selected_amb["y"] - incident["y"])**2)
        drive_time = dist / 15.0 
        
        # 2. Calculate Hospital Wait Time (Assign to nearest hospital for this simplified step)
        target_hospital = self.hospitals[action_idx % 8] # Rough mapping to base hospital
        target_hospital.admit_patient()
        wait_time = target_hospital.estimate_wait_time()
        
        # 3. Formulate Reward
        total_time_to_care = drive_time + wait_time
        reward = -total_time_to_care # Negative reward; agent wants to maximize this towards 0
        
        # Mark ambulance as busy (simplified logic: it becomes available again after 1 hour)
        self.fleet[action_idx]["available"] = 0.0 
        
        self.current_incident_idx += 1
        done = self.current_incident_idx >= len(self.incident_schedule)
        next_state = self._get_state()
        
        return next_state, reward, done

## 2. The Training Loop
We initialize the DQN Agent. The loop automatically checks for existing weights to resume training. We will run 500 episodes (simulation days). During each episode, the agent will handle all 30 daily incidents, storing the outcomes in its replay buffer to learn the optimal routing strategy.

In [3]:
# Hyperparameters
EPISODES = 500
BATCH_SIZE = 64
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.995

# Initialize Environment and Agent
env = EMSEnvironment(incidents_path)
state_dim = 47   # 36 fleet + 8 hospitals + 3 incident
action_dim = 12  # 12 ambulances to choose from

agent = DispatchAgent(state_dim=state_dim, action_dim=action_dim, lr=1e-3)

# Resumability: Load existing weights if they exist
agent.load_model(model_save_path)

epsilon = EPSILON_START
best_episode_reward = -float('inf')

print("\n--- Starting DQN Training Loop ---")
for episode in range(1, EPISODES + 1):
    state = env.reset()
    episode_reward = 0
    done = False
    
    while not done:
        # Create a boolean mask of available ambulances
        available_mask = [amb["available"] == 1.0 for amb in env.fleet]
        
        # If no ambulances are available, skip this incident (massive penalty could be applied here)
        if not any(available_mask):
            env.current_incident_idx += 1
            done = env.current_incident_idx >= len(env.incident_schedule)
            continue
            
        action = agent.select_action(state, epsilon, available_mask)
        next_state, reward, done = env.step(action)
        
        # Store experience in memory
        agent.memory.push(state, action, reward, next_state, done)
        state = next_state
        episode_reward += reward
        
        # Perform one step of gradient descent
        agent.update(BATCH_SIZE)

    # Sync target network every 10 episodes
    if episode % 10 == 0:
        agent.update_target_network()
        
    # Decay epsilon (reduce random exploration over time)
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    
    # Checkpointing: Save model if it achieved a new best reward
    if episode_reward > best_episode_reward:
        best_episode_reward = episode_reward
        agent.save_model(model_save_path)
        
    if episode % 50 == 0:
        print(f"Episode {episode}/{EPISODES} | Reward: {episode_reward:.2f} | Epsilon: {epsilon:.3f}")

print("--- Training Complete ---")

2026-03-24 11:22:01,555 - INFO - DQN initialized on device: mps
2026-03-24 11:22:02,251 - INFO - No existing checkpoint found. Initializing fresh weights.
2026-03-24 11:22:02,256 - INFO - Model saved to ../models/dqn_dispatch_v1.pt
2026-03-24 11:22:02,259 - INFO - Model saved to ../models/dqn_dispatch_v1.pt
2026-03-24 11:22:02,263 - INFO - Model saved to ../models/dqn_dispatch_v1.pt



--- Starting DQN Training Loop ---


2026-03-24 11:22:03,567 - INFO - Model saved to ../models/dqn_dispatch_v1.pt


Episode 50/500 | Reward: -4588.78 | Epsilon: 0.778
Episode 100/500 | Reward: -4759.95 | Epsilon: 0.606


2026-03-24 11:22:08,318 - INFO - Model saved to ../models/dqn_dispatch_v1.pt
2026-03-24 11:22:08,418 - INFO - Model saved to ../models/dqn_dispatch_v1.pt


Episode 150/500 | Reward: -4376.83 | Epsilon: 0.471
Episode 200/500 | Reward: -4676.52 | Epsilon: 0.367
Episode 250/500 | Reward: -4387.84 | Epsilon: 0.286
Episode 300/500 | Reward: -4579.82 | Epsilon: 0.222
Episode 350/500 | Reward: -4575.11 | Epsilon: 0.173
Episode 400/500 | Reward: -4563.26 | Epsilon: 0.135
Episode 450/500 | Reward: -4229.68 | Epsilon: 0.105
Episode 500/500 | Reward: -4495.28 | Epsilon: 0.082
--- Training Complete ---
